1: Tải dữ liệu gốc từ Kaggle

In [1]:
import kagglehub
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split

# Tải bộ dữ liệu
path = kagglehub.dataset_download("purusinghvi/email-spam-classification-dataset")
csv_path = os.path.join(path, "combined_data.csv")
df = pd.read_csv(csv_path)

print(f"Kích thước dữ liệu gốc: {df.shape[0]:,} dòng, {df.shape[1]} cột")

100%|██████████| 43.0M/43.0M [00:00<00:00, 157MB/s]

Extracting files...


Kích thước dữ liệu gốc: 83,448 dòng, 2 cột


2: Làm sạch dữ liệu

In [2]:
# 1. Xóa giá trị thiếu (nếu có)
df_clean = df.dropna(subset=['text', 'label']).copy()

# 2. Kiểm tra và xóa toàn bộ các dòng có nội dung trùng lặp nhưng khác nhãn (keep=False)
dup_mask = df_clean.duplicated(subset=['text'], keep=False)
print(f"Số dòng có nội dung trùng lặp cần loại bỏ: {dup_mask.sum()}")
df_clean = df_clean[~dup_mask]

# 3. Chuẩn hóa văn bản cơ bản (chuyển chữ thường, gom nhiều khoảng trắng thành 1)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['text'] = df_clean['text'].apply(clean_text)

# 4. Loại bỏ các dòng rỗng sau khi làm sạch (nếu có)
df_clean = df_clean[df_clean['text'] != ''].reset_index(drop=True)

print(f"Kích thước sau khi làm sạch: {df_clean.shape[0]:,} dòng")

Số dòng có nội dung trùng lặp cần loại bỏ: 4
Kích thước sau khi làm sạch: 83,444 dòng


3: Chia tập Train (80%) và Test (20%)

In [3]:
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['label']
)

print(f"Tập Train: {train_df.shape[0]:,} dòng")
print(f"Tập Test : {test_df.shape[0]:,} dòng")

print("\nTỷ lệ nhãn tập Train (%):")
print((train_df['label'].value_counts(normalize=True) * 100).round(2))

print("\nTỷ lệ nhãn tập Test (%):")
print((test_df['label'].value_counts(normalize=True) * 100).round(2))

Tập Train: 66,755 dòng
Tập Test : 16,689 dòng

Tỷ lệ nhãn tập Train (%):
label
1    52.62
0    47.38
Name: proportion, dtype: float64

Tỷ lệ nhãn tập Test (%):
label
1    52.62
0    47.38
Name: proportion, dtype: float64


4:Tải tập train và test

In [4]:
import shutil
from google.colab import files

# Tạo thư mục tạm để lưu
output_dir = "processed"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

# Nén thư mục processed thành file processed_data.zip
shutil.make_archive("processed_data", 'zip', output_dir)
print("Đã nén xong file processed_data.zip!")

# Tải trực tiếp về máy tính
files.download("processed_data.zip")

Đã nén xong file processed_data.zip!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>